# vLLM for LLM Serving

vLLM is a specialized serving engine for large language models. Where Triton is a general model-serving platform, vLLM focuses on text generation workloads: prompts, KV cache, streaming tokens, continuous batching, and OpenAI-compatible APIs.

This notebook shows how to think about vLLM deployment, how to start an OpenAI-compatible server, how to call it, and how to measure LLM-specific serving metrics.

## What you will learn

By the end, you should be able to explain and use:

- why LLM serving is different from ordinary model serving,
- prefill vs decode phases,
- KV cache memory pressure,
- PagedAttention at a high level,
- continuous batching,
- OpenAI-compatible vLLM serving,
- basic request benchmarking,
- and when to choose vLLM vs Triton.

## 1. Why vLLM exists

A classifier usually does one forward pass per request. An LLM request can require hundreds or thousands of forward passes because it generates tokens one at a time.

That creates special serving problems:

- requests have different prompt lengths,
- requests generate different numbers of output tokens,
- the KV cache can consume huge GPU memory,
- batching is dynamic because one request may finish while another continues,
- users often care about streaming and time-to-first-token.

vLLM is designed around these LLM-specific constraints.

## 2. Key serving concepts

| Concept | Meaning | Why it matters |
|---|---|---|
| Prefill | Process the prompt tokens and build KV cache | Long prompts are expensive before generation starts |
| Decode | Generate one new token at a time | Dominates long completions |
| KV cache | Stored keys/values for previous tokens | Saves compute but consumes memory |
| PagedAttention | vLLM memory-management approach for KV cache | Improves GPU memory utilization |
| Continuous batching | Dynamically batch active generation requests | Better throughput than static request batching |
| TTFT | Time to first token | Important for perceived responsiveness |
| TPOT | Time per output token | Important for streaming generation speed |
| Tokens/sec | Output token throughput | Useful capacity metric |

## 3. Setup

This notebook does not require vLLM to be installed unless you run the actual serving or offline-generation cells. The command-building and client examples are safe to run anywhere.

vLLM is usually used on Linux with NVIDIA GPUs. CPU-only use is possible in some configurations, but the main production path is GPU serving.

In [ ]:
import concurrent.futures
import json
import statistics
import textwrap
import time
from dataclasses import dataclass

import httpx
import pandas as pd

try:
    import torch
    cuda_available = torch.cuda.is_available()
    gpu_count = torch.cuda.device_count() if cuda_available else 0
except Exception:
    cuda_available = False
    gpu_count = 0

print(f"CUDA available: {cuda_available}")
print(f"GPU count: {gpu_count}")

## 4. Installation notes

Install vLLM only when you are ready to run an LLM server. The exact install command depends on CUDA, PyTorch, and your platform.

Typical starting point:

```bash
pip install vllm
```

For production or GPU-specific environments, use the official vLLM installation guidance for the matching CUDA/PyTorch versions.

In [ ]:
install_commands = {
    "basic_pip": "pip install vllm",
    "openai_client": "pip install openai httpx",
    "note": "Check vLLM docs for CUDA-specific wheels and production containers.",
}
install_commands

## 5. Choose a model

For learning, choose a small instruct model first. Larger models need more GPU memory.

Examples:

| Model | Why use it | Notes |
|---|---|---|
| `HuggingFaceTB/SmolLM2-135M-Instruct` | Tiny learning model | Good for API plumbing, not quality |
| `Qwen/Qwen2.5-0.5B-Instruct` | Small but more capable | Better quality, still modest |
| `TinyLlama/TinyLlama-1.1B-Chat-v1.0` | Classic small LLM demo | Needs more memory |
| `mistralai/Mistral-7B-Instruct-v0.3` | Real serving target | Needs a capable GPU |
| `meta-llama/Llama-3.1-8B-Instruct` | Common production-style target | May require gated model access |

In [ ]:
@dataclass
class VLLMServingPlan:
    model: str = "HuggingFaceTB/SmolLM2-135M-Instruct"
    host: str = "0.0.0.0"
    port: int = 8000
    dtype: str = "auto"
    max_model_len: int = 2048
    gpu_memory_utilization: float = 0.85
    tensor_parallel_size: int = 1
    max_num_seqs: int = 128

plan = VLLMServingPlan()
plan

## 6. Start an OpenAI-compatible vLLM server

The most common vLLM deployment shape is an OpenAI-compatible HTTP server. That means existing OpenAI client code can point at your local or remote vLLM base URL.

The command below is generated from the serving plan. Run it in a terminal when you are ready to start the server.

In [ ]:
def build_vllm_serve_command(plan):
    return textwrap.dedent(f"""
    vllm serve {plan.model} \\n
      --host {plan.host} \\n
      --port {plan.port} \\n
      --dtype {plan.dtype} \\n
      --max-model-len {plan.max_model_len} \\n
      --gpu-memory-utilization {plan.gpu_memory_utilization} \\n
      --tensor-parallel-size {plan.tensor_parallel_size} \\n
      --max-num-seqs {plan.max_num_seqs}
    """).strip()

print(build_vllm_serve_command(plan))

## 7. Docker server command

A container is often cleaner for serving. The exact vLLM image tag changes over time, so treat this as a template.

Important flags:

- `--gpus all`: expose GPUs to the container.
- `--ipc=host`: helps shared-memory behavior for PyTorch workloads.
- Hugging Face cache mount: avoids redownloading models every run.
- `HF_TOKEN`: needed for gated models such as some Llama checkpoints.

In [ ]:
docker_command = f"""
docker run --rm --gpus all --ipc=host \
  -p {plan.port}:{plan.port} \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  -e HF_TOKEN=$HF_TOKEN \
  vllm/vllm-openai:latest \
  --model {plan.model} \
  --host {plan.host} \
  --port {plan.port} \
  --dtype {plan.dtype} \
  --max-model-len {plan.max_model_len}
""".strip()
print(docker_command)

## 8. Health checks

After the server starts, check health and model listing. vLLM exposes an OpenAI-compatible API, so `/v1/models` is the main sanity check.

In [ ]:
base_url = f"http://localhost:{plan.port}/v1"
health_urls = [
    f"{base_url}/models",
]
for url in health_urls:
    try:
        response = httpx.get(url, timeout=2.0)
        print(f"{url} -> {response.status_code}")
        print(response.text[:500])
    except Exception as error:
        print(f"{url} skipped or failed: {error}")

## 9. Call vLLM with the OpenAI Python client

Because vLLM can expose OpenAI-compatible endpoints, you can reuse `openai` client code. The API key can be any non-empty string for a local unauthenticated server.

This cell skips gracefully if the server is not running.

In [ ]:
try:
    from openai import OpenAI

    client = OpenAI(base_url=base_url, api_key="not-needed")
    response = client.chat.completions.create(
        model=plan.model,
        messages=[
            {"role": "system", "content": "You are a concise ML systems tutor."},
            {"role": "user", "content": "Explain KV cache in one paragraph."},
        ],
        temperature=0.3,
        max_tokens=120,
    )
    print(response.choices[0].message.content)
except Exception as error:
    print("OpenAI-compatible request skipped. Start vLLM first, then rerun this cell.")
    print(f"Reason: {error}")

## 10. Raw HTTP request

The OpenAI client is convenient, but raw HTTP makes the request shape obvious. This is also useful when writing load tests.

In [ ]:
chat_payload = {
    "model": plan.model,
    "messages": [
        {"role": "system", "content": "You are a concise ML systems tutor."},
        {"role": "user", "content": "What is continuous batching in vLLM?"},
    ],
    "temperature": 0.3,
    "max_tokens": 120,
}
print(json.dumps(chat_payload, indent=2))

In [ ]:
try:
    response = httpx.post(f"{base_url}/chat/completions", json=chat_payload, timeout=30.0)
    print(f"Status: {response.status_code}")
    print(response.text[:1200])
except Exception as error:
    print("Raw HTTP request skipped. Start vLLM first, then rerun this cell.")
    print(f"Reason: {error}")

## 11. Offline batch generation with vLLM

vLLM can also run offline batch generation from Python. This is useful for evaluation, synthetic data generation, and batch jobs.

The cell below runs only if vLLM is installed and the selected model can fit in your environment.

In [ ]:
prompts = [
    "Explain prefill vs decode in LLM serving.",
    "Why does KV cache use so much GPU memory?",
    "When would you choose vLLM over Triton?",
]

try:
    from vllm import LLM, SamplingParams

    sampling_params = SamplingParams(temperature=0.3, max_tokens=100)
    llm = LLM(model=plan.model, max_model_len=plan.max_model_len)
    outputs = llm.generate(prompts, sampling_params)

    for output in outputs:
        print("=" * 80)
        print(output.prompt)
        print(output.outputs[0].text)
except Exception as error:
    print("Offline vLLM generation skipped. Install vLLM and choose a model that fits your hardware.")
    print(f"Reason: {error}")

## 12. Benchmarking an LLM endpoint

LLM serving benchmarks should track more than request latency. Useful metrics include:

- request latency,
- time to first token if streaming,
- output tokens per second,
- prompt tokens per second,
- total tokens per second,
- error rate,
- concurrency,
- p50/p95/p99 latency.

This simple benchmark measures non-streaming latency and estimates output tokens from the API usage field when available.

In [ ]:
def percentile(values, q):
    values = sorted(values)
    if not values:
        return float("nan")
    index = min(len(values) - 1, max(0, round((q / 100) * (len(values) - 1))))
    return values[index]

def one_chat_request(prompt):
    payload = {
        "model": plan.model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 100,
    }
    start = time.perf_counter()
    response = httpx.post(f"{base_url}/chat/completions", json=payload, timeout=60.0)
    elapsed = time.perf_counter() - start
    data = response.json() if response.status_code == 200 else {}
    usage = data.get("usage", {})
    output_tokens = usage.get("completion_tokens", 0)
    total_tokens = usage.get("total_tokens", 0)
    return {
        "status": response.status_code,
        "latency_s": elapsed,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
    }

def run_endpoint_benchmark(total_requests=12, concurrency=3):
    benchmark_prompts = [
        "Explain dynamic batching in one paragraph.",
        "Give three causes of high p99 latency in LLM serving.",
        "What is the role of KV cache?",
        "Compare vLLM and Triton briefly.",
    ]
    prompts = [benchmark_prompts[i % len(benchmark_prompts)] for i in range(total_requests)]
    start = time.perf_counter()
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [executor.submit(one_chat_request, prompt) for prompt in prompts]
        for future in concurrent.futures.as_completed(futures):
            results.append(future.result())
    total_time = time.perf_counter() - start

    latencies = [item["latency_s"] * 1000 for item in results]
    output_tokens = sum(item["output_tokens"] for item in results)
    total_tokens = sum(item["total_tokens"] for item in results)
    successes = sum(item["status"] == 200 for item in results)

    return {
        "requests": total_requests,
        "concurrency": concurrency,
        "successes": successes,
        "requests_per_sec": total_requests / total_time,
        "output_tokens_per_sec": output_tokens / total_time if output_tokens else 0,
        "total_tokens_per_sec": total_tokens / total_time if total_tokens else 0,
        "mean_latency_ms": statistics.mean(latencies),
        "p50_latency_ms": percentile(latencies, 50),
        "p95_latency_ms": percentile(latencies, 95),
        "p99_latency_ms": percentile(latencies, 99),
    }

try:
    pd.DataFrame([run_endpoint_benchmark(total_requests=8, concurrency=2)])
except Exception as error:
    print("Endpoint benchmark skipped. Start vLLM first, then rerun this cell.")
    print(f"Reason: {error}")

## 13. Streaming and TTFT

For chat products, users care about how quickly the first token appears. That is TTFT: time to first token.

A request can have good total latency but poor TTFT, especially with long prompts. Streaming lets the user see partial output while the model continues decoding.

In [ ]:
try:
    from openai import OpenAI

    client = OpenAI(base_url=base_url, api_key="not-needed")
    start = time.perf_counter()
    first_token_time = None
    chunks = []

    stream = client.chat.completions.create(
        model=plan.model,
        messages=[{"role": "user", "content": "Explain time to first token in LLM serving."}],
        max_tokens=120,
        temperature=0.2,
        stream=True,
    )

    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        if delta and first_token_time is None:
            first_token_time = time.perf_counter() - start
        chunks.append(delta)

    total_time = time.perf_counter() - start
    print(f"TTFT: {first_token_time:.3f}s")
    print(f"Total time: {total_time:.3f}s")
    print("".join(chunks))
except Exception as error:
    print("Streaming example skipped. Start vLLM first, then rerun this cell.")
    print(f"Reason: {error}")

## 14. Tuning knobs

Common vLLM tuning parameters:

| Parameter | What it controls | Typical trade-off |
|---|---|---|
| `--max-model-len` | Maximum context length | Larger context uses more KV cache memory |
| `--gpu-memory-utilization` | Fraction of GPU memory vLLM may use | Higher can improve capacity but risks OOM |
| `--max-num-seqs` | Max active sequences | Higher concurrency but more memory pressure |
| `--tensor-parallel-size` | Split model across GPUs | Enables larger models but adds communication |
| `--dtype` | Weight/activation dtype | Lower precision saves memory |
| `--quantization` | Quantized weights | Saves memory, may affect quality/speed |
| `--enable-prefix-caching` | Cache shared prompt prefixes | Useful for repeated system prompts |

In [ ]:
tuning_table = pd.DataFrame([
    {"goal": "Lower memory", "try": "reduce max_model_len, use lower dtype, use quantization"},
    {"goal": "Higher throughput", "try": "increase max_num_seqs, tune concurrency, use faster dtype"},
    {"goal": "Lower TTFT", "try": "shorter prompts, prefix caching, avoid overload"},
    {"goal": "Serve larger model", "try": "tensor parallelism, quantization, more GPUs"},
    {"goal": "Improve tail latency", "try": "cap concurrency, reduce context length, monitor queueing"},
])
tuning_table

## 15. vLLM vs Triton

Use this mental model:

| Question | Prefer vLLM | Prefer Triton |
|---|---|---|
| Is the workload text generation from decoder-only LLMs? | Yes | Sometimes |
| Do you need OpenAI-compatible chat/completions quickly? | Yes | Not directly |
| Do you need general model serving for classifiers, embeddings, ONNX, ensembles? | Not usually | Yes |
| Is continuous batching for generation the main bottleneck? | Yes | Not usually |
| Do you need TensorRT, ONNX Runtime, Python backend, multi-model repo support? | Maybe | Yes |
| Are you serving one or more Hugging Face LLMs? | Often | Possible, but more setup |

In practice, teams often use both: vLLM for LLM generation and Triton for general ML inference.

## 16. Production checklist

Before calling a vLLM service production-ready, answer these:

- What model version is deployed?
- What is the max context length?
- What is the expected request rate and concurrency?
- What are p50/p95/p99 latency targets?
- What are TTFT and tokens/sec targets?
- What happens on GPU OOM?
- Are prompts and outputs logged safely?
- Are model access tokens stored outside code?
- Is there rate limiting?
- Is there monitoring for queueing, errors, and GPU memory?
- Is there a rollback plan?

## 17. Exercises

1. Start vLLM with a small model and call `/v1/models`.
2. Send one chat request with the OpenAI client.
3. Run the benchmark at concurrency 1, 2, 4, and 8. Plot p95 latency and tokens/sec.
4. Compare streaming TTFT for a short prompt vs a long prompt.
5. Reduce `max_model_len` and observe GPU memory usage.
6. Try `--enable-prefix-caching` with repeated system prompts.
7. Write a short decision note: vLLM, Triton, or both for your target use case.

## Summary

vLLM is one of the main tools used to deploy LLMs. It is especially strong for high-throughput text generation because it is built around KV cache management and continuous batching.

What you now have:

- a vLLM serving command template,
- a Docker serving template,
- OpenAI-compatible client examples,
- raw HTTP request examples,
- offline generation structure,
- endpoint benchmark helpers,
- streaming and TTFT measurement structure,
- and a clear comparison with Triton.

This completes the LLM-specific serving branch of Week 3-4.